In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git
!pip install git+https://github.com/facebookresearch/segment-anything.git

#**DATASET**

In [17]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import cv2

from sklearn.model_selection import train_test_split
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
import torch.optim as optim

import clip
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator

import copy
import math
import random
from typing import Dict, List, Optional, Tuple

In [13]:
class AMLDataset(Dataset):

    def __init__(self, csv_path, imgs_dir, train=True, transform=None):
        self.imgs_dir = imgs_dir
        self.train = train
        self.transform = transform

        full_df = pd.read_csv(csv_path)

        train_df, test_df = train_test_split(
            full_df,
            test_size=0.20,            # 20% for validation
            random_state=42
        )

        self.df = (train_df if train else test_df).reset_index(drop=True)

        # Create the 'classes' attribute (Unique list of names)
        # We sort them to ensure the index mapping is always consistent
        self.classes = sorted(self.df['label'].unique().tolist())

        # Create a mapping from Name -> Integer ID
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # Create the 'labels' attribute (The ID for every single row)
        # This is helpful if you want to use a Weighted or Balanced Sampler later
        self.labels = [self.class_to_idx[name] for name in self.df['label']]


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_img(row['filename'])
        return img, self.labels[idx]

    def _load_img(self, filename):
        path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img




class AMLValidation(Dataset):
    def __init__(self, csv_path, imgs_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.imgs_dir = imgs_dir
        self.transform = transform

        # Group the rows by episode_id so __len__ returns total number of tasks
        self.episode_ids = sorted(self.df['episode_id'].unique())

    def __len__(self):
        return len(self.episode_ids)

    def load_img(self, filename):
        img_path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        ep_id = self.episode_ids[idx]
        ep_df = self.df[self.df['episode_id'] == ep_id]

        support_df = ep_df[ep_df['role'] == 'support']
        query_df = ep_df[ep_df['role'] == 'query']

        # Convertiamo subito le immagini in tensori PyTorch!
        s_imgs = torch.stack([self.load_img(f) for f in support_df['filename']])
        s_labels = support_df['label'].values

        q_imgs = torch.stack([self.load_img(f) for f in query_df['filename']])

        # Gestiamo il caso in cui le label delle query siano nascoste per il test
        q_labels = query_df['label'].values if 'label' in query_df.columns else None

        return {
            "support_imgs": s_imgs,
            "support_labels": s_labels,
            "query_imgs": q_imgs,
            "query_labels": q_labels,
            "episode_id": ep_id
        }



class SAMModel:

    def __init__(self, model_type = 'vit_b', check_point_path = "./drive/MyDrive/AdvancedML/Project/sam_vit_b_01ec64.pth"):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = sam_model_registry[model_type](check_point_path)
        self.model.to(device=self.device)
        self.predictor = SamPredictor(self.model)
        self.mask_generator = SamAutomaticMaskGenerator(self.model)


    def predict_mask(self, img, input_point, input_label):
        self.predictor.set_image(img)
        masks, _, _ = self.predictor.predict(point_coords=input_point,
                                             point_labels=input_label)

    def generate_all_masks(self, img):

        masks = self.mask_generator.generate(img)
        return masks

In [ ]:
CSV_PATH  = "./drive/MyDrive/AdvancedML/Project/release/train.csv"
IMGS_DIR  = "./drive/MyDrive/AdvancedML/Project/release/images"
OUTPUT_CSV_PATH = "./drive/MyDrive/AdvancedML/Project/release/train_augmented.csv"


def aug_with_sam(sam: SAMModel, train_csv_path: str, train_imgs_path: str, output_csv_path: str):

    df_originale = pd.read_csv(train_csv_path)

    nuove_righe = [] # a questa lista aggiungo piano piano le nuove righe che andrò a inserire nel df originale (train.csv)

    filenames = sorted(os.listdir(IMGS_DIR), key=lambda x: int(x.split('_')[1].split('.')[0])) # ordino i filename per numero di immagine

    for fname in filenames:

      # Estraggo il numero dell'immagine
      split1 = fname.split('_')[1]
      split2 = split1.split('.')[0]
      num_img = int(split2)

      # Per considerare solo le immagini di train (num_im > 0 & num_img <= 5000)
      if num_img > 5000:
            continue

      # Estraggo la riga del csv corrispondente al filename
      riga_match = df_originale[df_originale['filename'] == fname]
      if riga_match.empty:
            continue # Se il file non è nel CSV, lo saltiamo

      label = riga_match.iloc[0]['label']
      print(f"Elaborazione file: {fname} (Label: {label})...")
      img_path = os.path.join(IMGS_DIR, fname)
      Im = Image.open(img_path).convert('RGB')
      img = np.array(Im)

      masks = sam.generate_all_masks(img)
      if not masks:
        continue

      h,w, _ = img.shape
      area_totale = h * w
      maschera_piu_grande = max(masks, key=lambda x: x['area'])


      for idx, mask_dict in enumerate(masks):
        area_px = mask_dict['area']
        percentuale_area = area_px / area_totale
        bbox = mask_dict['bbox']  # [x, y, w, h]

        # --- CASO 1: Troppo piccola (Rumore) ---
        if percentuale_area < 0.05:
          continue

        # --- CASO 2: La più grande (Contesto) ---
        if mask_dict is maschera_piu_grande:
          pass

        # --- CASO 3: Nella media (Soggetto/Crop) ---
        if 0.01 <= percentuale_area <= 0.85:
          x, y, w, h = map(int, bbox)

          # Ritaglio (Crop) basato sulla bounding box di SAM
          crop_soggetto = img[y : y+h, x : x+w]

          # Evitiamo crop falliti o vuoti
          if crop_soggetto.size == 0:
              continue

          # Resize a 224x224 per CLIP
          crop_resized = cv2.resize(crop_soggetto, (224, 224))

          # Nome univoco per il crop
          nuovo_fname = f"aug_crop_{idx}_{fname}"
          nuovo_path = os.path.join(IMGS_DIR, nuovo_fname)

          # Salviamo l'immagine
          cv2.imwrite(nuovo_path, cv2.cvtColor(crop_resized, cv2.COLOR_RGB2BGR))

          # Prepariamo la nuova riga per il CSV
          nuove_righe.append({'filename': nuovo_fname, 'label': label})

      # Creiamo un DataFrame con tutte le nuove immagini generate
      df_nuovo = pd.DataFrame(nuove_righe)

      # Uniamo il vecchio DataFrame con quello nuovo (concatenazione verticale)
      df_finale = pd.concat([df_originale, df_nuovo], ignore_index=True)

    # Salviamo il nuovo CSV aggiornato
    df_finale.to_csv(output_csv_path, index=False)
    print(f"\nAugmentation completata! Nuovo CSV salvato in: {output_csv_path}")
    print(f"Immagini totali originali: {len(df_originale)}")
    print(f"Nuove immagini aggiunte: {len(df_nuovo)}")
    print(f"Totale immagini nel nuovo dataset: {len(df_finale)}")


# Avvia il processo completo
sam = SAMModel()
aug_with_sam(sam, CSV_PATH, IMGS_DIR, OUTPUT_CSV_PATH)

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', DEVICE, jit=False)
model.eval()
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Define constants
LR = 1e-4 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

CSV_PATH  = "drive/MyDrive/AdvancedML/Project/release/train_augmented.csv"
IMGS_DIR  = "drive/MyDrive/AdvancedML/Project/release//images/"
VAL_CSV   = "drive/MyDrive/AdvancedML/Project/release/test_episodes_release.csv"

# Download the dataset
train_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=False,
    transform=preprocess
)

val_dataset = AMLValidation(
    csv_path=VAL_CSV,
    imgs_dir=IMGS_DIR,
    transform=preprocess
)

# Prepare the data
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_train = len(train_loader)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_test = len(test_loader)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_val = len(val_loader)

#**ADAPTATION**

In [16]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0


def set_peft_seed(seed: int = 0) -> None:
    """Make the lab results more reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_peft_seed(SEED)
print("PEFT device:", DEVICE)


PEFT device: cpu


In [ ]:
class CLIPLinearAdapter(nn.Module):
    def __init__(self, clip_model, embed_dim, num_classes):
        super(CLIPLinearAdapter, self).__init__()
        self.clip_model = clip_model

        # Congeliamo i pesi nativi di CLIP
        for param in self.clip_model.parameters():
            param.requires_grad = False

        # Strato di Adattamento Lineare (l'unica parte che si addestra)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, image_inputs):
        # Estraiamo le feature con CLIP
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image_inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Classificazione
        logits = self.classifier(image_features.float())
        return logits, image_features

num_classes = len(train_dataset.classes)
# Dimensione dell'embedding per ViT-B/32 è 512
adapter_model = CLIPLinearAdapter(model, embed_dim=512, num_classes=num_classes).to(DEVICE)

# Definiamo come calcolare l'errore e come ottimizzare
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(adapter_model.classifier.parameters(), lr=LR, weight_decay=1e-4)

In [ ]:
print(f"Avvio addestramento del modulo CLIP Adaptation per {EPOCHS} epoche...")

for epoch in range(EPOCHS):
    adapter_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        logits, _ = adapter_model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    print(f"Epoca [{epoch+1}/{EPOCHS}] -> Loss: {epoch_loss:.4f} | Accuratezza Train: {epoch_acc:.2f}%")

In [ ]:
adapter_model.eval()

print("Avvio validazione episodica per la Submission...")

all_episode_predictions = []
query_id = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="[Generazione Submission]"):

        # 1. Estraiamo le immagini dell'episodio
        support_imgs = batch["support_imgs"].squeeze(0).to(DEVICE)
        support_labels = batch["support_labels"][0]
        query_imgs = batch["query_imgs"].squeeze(0).to(DEVICE)

        # 2. Otteniamo le feature visive usando il NOSTRO Adapter
        _, s_adapted_features = adapter_model(support_imgs)
        _, q_adapted_features = adapter_model(query_imgs)

        # 3. Costruiamo i prototipi dell'episodio per ogni classe
        unique_episode_labels = np.unique(support_labels)
        episode_prototypes = []

        for label in unique_episode_labels:
            indices = [i for i, l in enumerate(support_labels) if l == label]
            prototype = s_adapted_features[indices].mean(dim=0)
            prototype = prototype / prototype.norm(dim=-1, keepdim=True)
            episode_prototypes.append(prototype)

        episode_prototypes = torch.stack(episode_prototypes)

        # 4. Classifichiamo le Query Images calcolando la similarità
        similarity = q_adapted_features @ episode_prototypes.T
        predictions_indices = similarity.argmax(dim=-1).cpu().numpy()

        # 5. Mappiamo le predizioni e prepariamo il salvataggio
        for idx in predictions_indices:
            pred_label = unique_episode_labels[idx]
            all_episode_predictions.append({
                "Id": query_id,
                "label": pred_label
            })
            query_id += 1

# ==========================================
# SALVATAGGIO DEL CSV FINALE
# ==========================================
df_submission = pd.DataFrame(all_episode_predictions)
csv_filename = "group_3_submission.csv"
df_submission.to_csv(csv_filename, index=False)

print(f"\nOperazione completata! Predizioni salvate in '{csv_filename}'.")
print(f"Totale query analizzate: {len(df_submission)}")